In [1]:
import pandas as pd

# Load the true and fake news datasets
print("Loading datasets...")
df_true = pd.read_csv("../data/True.csv")
df_fake = pd.read_csv("../data/Fake.csv")

# Assign target labels (1 for Real, 0 for Fake)
df_true['label'] = 1
df_fake['label'] = 0

# Combine both dataframes into a single dataset
df = pd.concat([df_true, df_fake], ignore_index=True)

# Shuffle the rows to ensure a random distribution
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Inspect dataset structure and class balance
print(df.info())
print("\nClass Distribution (1 = Real, 0 = Fake):")
print(df['label'].value_counts())

# Display the first few rows
display(df.head())

Loading datasets...
<class 'pandas.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    44898 non-null  str  
 1   text     44898 non-null  str  
 2   subject  44898 non-null  str  
 3   date     44898 non-null  str  
 4   label    44898 non-null  int64
dtypes: int64(1), str(4)
memory usage: 112.3 MB
None

Class Distribution (1 = Real, 0 = Fake):
label
0    23481
1    21417
Name: count, dtype: int64


,title,text,subject,date,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",0
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",0
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",0
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",1
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",0


In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # 1. Convert to lowercase
    text = text.lower()
    
    # 2. Remove punctuation and numbers (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 3. Split into words (tokenization)
    words = text.split()
    
    # 4. Remove stop words and apply lemmatization
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    # 5. Join words back into a single string
    return ' '.join(cleaned_words)

# Apply cleaning to a small sample first to test (e.g., first 1000 rows)
# Note: Cleaning the entire dataset of 40k+ rows might take a few minutes.
print("Cleaning text data... (this may take a moment)")
df['cleaned_text'] = df['text'].head(5000).apply(clean_text)

print("Text cleaning completed for sample rows!")
display(df[['text', 'cleaned_text', 'label']].head(3))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Welcome\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Welcome\AppData\Roaming\nltk_data...


Cleaning text data... (this may take a moment)
Text cleaning completed for sample rows!


,text,cleaned_text,label
0,"Donald Trump s White House is in chaos, and th...",donald trump white house chaos trying cover ru...,0
1,Now that Donald Trump is the presumptive GOP n...,donald trump presumptive gop nominee time reme...,0
2,Mike Pence is a huge homophobe. He supports ex...,mike penny huge homophobe support exgay conver...,0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Drop rows where cleaned_text is NaN (since we only cleaned the first 5000 rows for testing)
df_model = df.dropna(subset=['cleaned_text']).copy()

# 1. Define features (X) and target (y)
X = df_model['cleaned_text']
y = df_model['label']

# 2. Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and fit the TF-IDF Vectorizer
print("Vectorizing text using TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 4. Train Logistic Regression Model
print("Training Logistic Regression model...")
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

# 5. Evaluate the Model
y_pred = model.predict(X_test_tfidf)

print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Vectorizing text using TF-IDF...
Training Logistic Regression model...

--- Model Evaluation ---
Accuracy: 96.90%

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       533
           1       0.95      0.98      0.97       467

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000



In [4]:
# 1. Clean the entire dataset
print("Cleaning the full dataset (this may take a couple of minutes)...")
df['cleaned_text'] = df['text'].apply(clean_text)

# 2. Prepare features and target
X = df['cleaned_text']
y = df['label']

# 3. Train/Test Split (80% train, 20% test)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Vectorize with TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
print("Vectorizing full text using TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 5. Train Logistic Regression Model
from sklearn.linear_model import LogisticRegression
print("Training model on the full dataset...")
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

# 6. Evaluate the Model
from sklearn.metrics import accuracy_score, classification_report
y_pred = model.predict(X_test_tfidf)

print(f"\n--- Full Dataset Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 7. Save model and vectorizer using pickle for Streamlit
import pickle
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))
print("\nSuccess! Model and vectorizer saved as 'model.pkl' and 'vectorizer.pkl'.")

Cleaning the full dataset (this may take a couple of minutes)...
Vectorizing full text using TF-IDF...
Training model on the full dataset...

--- Full Dataset Model Evaluation ---
Accuracy: 98.72%

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4669
           1       0.98      0.99      0.99      4311

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980


Success! Model and vectorizer saved as 'model.pkl' and 'vectorizer.pkl'.
